# 📈 SentimentArcs: Plotting the Emotional Journey of a Novel

**Updated & Refactored (Oct 2025)**

This notebook is a pedagogical refactor of the original [SentimentArcs Simplified Notebook (Oct 2022)](https://github.com/jon-chun/sentimentarcs_notebooks) by Jon Chun. 

This version is designed to teach non-STEM students the core concepts of diachronic sentiment analysis by focusing on a clean, linear workflow and modern, efficient code.

---

## 💡 Introduction: What is a "Sentiment Arc"?

Have you ever felt the flow of a story? The way it builds tension, hits a low point, and then rises to a hopeful conclusion? A **sentiment arc** is a way to visualize that emotional journey.

**Our Goal:** We will write a program to read an entire novel, break it into individual sentences, and then score *every single sentence* for its emotional content (from negative to positive). 

Finally, we'll plot that score from the novel's beginning to its end, creating a data-driven map of the story's emotional shape.



---

## ⚙️ Step 0: Setup & Installation

First, we need to install the libraries that do the heavy lifting. We'll also check if Google Colab has given us a **GPU**, which makes our AI models *much* faster.

In [ ]:
# Check for GPU
!nvidia-smi

In [ ]:
%pip install -q transformers vaderSentiment spacy plotly scipy
!python -m spacy download en_core_web_sm

In [ ]:
# Import all our libraries
import pandas as pd
import numpy as np
import spacy
import plotly.express as px
from google.colab import files
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline
from scipy.signal import savgol_filter
from tqdm.notebook import tqdm
import torch
import warnings

# Suppress warnings for a cleaner notebook
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

---

## 📚 Step 1: Load Your Novel

Run the cell below to upload a plain text (`.txt`) file of a novel. 

You can get copyright-free books from sites like [Project Gutenberg](https://www.gutenberg.org/).

In [ ]:
print("Please upload your novel's .txt file:")
uploaded = files.upload()

# Get the file name and content
novel_name_str = ""
novel_raw_str = ""

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')
  novel_name_str = fn
  # Decode the raw bytes into a string
  try:
    novel_raw_str = uploaded[novel_name_str].decode('utf-8')
  except UnicodeDecodeError:
    print("UTF-8 decoding failed. Trying 'latin-1'...")
    novel_raw_str = uploaded[novel_name_str].decode('latin-1')

print("\n--- Sample of your novel: ---")
print(novel_raw_str[1000:1500])
print("------------------------------")

---

## 📖 Step 2: Prepare the Text with `spaCy`

A novel is just one giant string of text. To analyze sentiment, we need to break it into sentence-by-sentence chunks. 

We'll use a powerful library called **`spaCy`** to do this. `spaCy` is smart enough to find sentence boundaries, even with tricky punctuation like "Mr. Smith" or "p. 55." This is called **sentence segmentation**.

In [ ]:
# Load the small English spaCy model
nlp = spacy.load("en_core_web_sm")

# Increase the max length to handle a whole novel
nlp.max_length = len(novel_raw_str) + 100

print(f"Processing '{novel_name_str}' with spaCy... (This may take a minute)")
doc = nlp(novel_raw_str)

# Extract sentences
# We also clean them up by stripping whitespace and replacing newlines
sentences = [sent.text.strip().replace('\n', ' ').replace('\r', ' ') for sent in doc.sents]

# Filter out any empty sentences that might result from page breaks, etc.
sentences = [s for s in sentences if s]

print(f"\nSuccessfully segmented the novel into {len(sentences)} sentences.")

In [ ]:
# Create our main DataFrame to store all our data
sentiment_df = pd.DataFrame({'line': sentences})
sentiment_df['line_no'] = sentiment_df.index

# Reorder columns to be tidy
sentiment_df = sentiment_df[['line_no', 'line']]

print("Here are the first 5 sentences:")
sentiment_df.head()

---

## 🤔 Step 3: The "Sentiment" in Sentiment Analysis

Now that we have our sentences, how do we get a 'sentiment score' for each one? We'll try two major approaches: one simple and fast, the other complex and 'smart'.

### Method 1: Lexicon-Based (A "Word Dictionary")

* **Analogy:** Imagine a "price list" for words. `good` = +2, `bad` = -2, `love` = +3, `hate` = -3. 
* **How it works:** This model (we'll use **VADER**) scans a sentence, looks up each word in its dictionary, and adds up the scores. It also knows a few simple rules (e.g., "not good" is negative, and "VERY good" is *more* positive).
* **Pros:** ✅ Extremely fast. Simple to understand.
* **Cons:** ❌ It's "dumb" and easily misses context or sarcasm. For VADER, the sentences "This is not bad" and "This is bad" might get similar negative scores because it gets confused by the word "bad".

### Method 2: Transformer-Based (An "AI Model")

* **Analogy:** A "mini-brain" (called a **Transformer**) that was trained by reading billions of sentences from the internet. We'll use a model called `siebert/sentiment-roberta-large-english`.
* **How it works:** Instead of just adding up words, this model reads the *entire sentence* at once to understand the relationships between words. 
* **Pros:** ✅ Very "smart" and understands context. It correctly knows that "not bad" is a positive sentiment.
* **Cons:** ❌ Much slower. It requires a GPU to run efficiently.

### 3a. Running Method 1 (VADER)

In [ ]:
print("Running VADER (Lexicon) analysis...")

# 1. Initialize the VADER sentiment object
sid_obj = SentimentIntensityAnalyzer()

# 2. Use .apply() to run VADER on every row in our 'line' column.
# We only want the 'compound' score, which is a single number from -1.0 (most negative) to +1.0 (most positive).
sentiment_df['vader'] = sentiment_df['line'].apply(lambda x: sid_obj.polarity_scores(x)['compound'])

print("VADER analysis complete.")
sentiment_df.head()

### 3b. Running Method 2 (Transformer)

In [ ]:
# Check for GPU and set the device
device = 0 if torch.cuda.is_available() else -1

if device == 0:
    print("✅ GPU found! Running on GPU.")
else:
    print("⚠️ No GPU found. Running on CPU (this will be much slower).")

# 1. Initialize the Transformer pipeline
# This model is robust and was in the original notebook
model_name = "siebert/sentiment-roberta-large-english"
print(f"Loading model '{model_name}'... (This may take a moment)")
sa_pipe = pipeline("sentiment-analysis", 
                   model=model_name, 
                   device=device)

# 2. Get all sentences as a list
all_sentences = sentiment_df['line'].to_list()

# 3. Run the pipeline on the *entire list* at once.
# This is MUCH faster as it batches the data for the GPU.
# We'll use a batch_size to help manage memory and show a progress bar.
print(f"Running Transformer analysis on {len(all_sentences)} sentences...")
transformer_results = []
# Use tqdm for a progress bar
for result in tqdm(sa_pipe(all_sentences, truncation=True, batch_size=128), total=len(all_sentences)):
    transformer_results.append(result)

print("Analysis complete.")

# 4. Process the results and add to our main DataFrame
# The result is a dict like {'label': 'POSITIVE', 'score': 0.99}
# We need to convert this to a single number: POSITIVE = +score, NEGATIVE = -score
def process_result(res):
    score = res['score']
    if res['label'] == 'NEGATIVE':
        return -score
    return score

sentiment_df['roberta'] = [process_result(r) for r in transformer_results]
sentiment_df.head()

---

##  smoothing Step 4: The "Arc" in Sentiment Arc (Smoothing the Data)

If we plot the raw sentiment scores for every sentence, the plot will look like a chaotic seismograph—it's too "noisy" to see the real story.

We need to **smooth** the data to find the *trend*.

* **Analogy:** You wouldn't plot the temperature every single second to see the change of seasons. You'd plot the 30-day average. Smoothing does the same thing for our novel.

### 4a. Smoothing Method 1: Rolling Mean

This is the simplest method. For each sentence, it calculates the average sentiment of the sentences around it (e.g., the 500 before and 500 after).

#### What is a "Hyperparameter"?

The `WINDOW_PERC` below is a **hyperparameter**—a setting *we* choose. 

* A **small** window (e.g., `0.01` or 1%) will create a **spiky, detailed** plot.
* A **large** window (e.g., `0.10` or 10%) will create a **smooth, simple** plot.

There is no "right" answer. Try changing it and see how the final plot changes!

In [ ]:
# --- Hyperparameter --- 
# What percentage of the novel to use for the smoothing window?
WINDOW_PERC = 0.05 # 5% of the book
# ------------------------

# Calculate window size in sentences
win_size = int(len(sentiment_df) * WINDOW_PERC)

# Ensure it's an odd number (important for centering the window)
if win_size % 2 == 0:
    win_size += 1 

print(f"Using a rolling mean window size of: {win_size} sentences")

# Apply the rolling mean
sentiment_df['vader_smooth_mean'] = sentiment_df['vader'].rolling(window=win_size, center=True, min_periods=1).mean()
sentiment_df['roberta_smooth_mean'] = sentiment_df['roberta'].rolling(window=win_size, center=True, min_periods=1).mean()

sentiment_df.head()

### 4b. Smoothing Method 2: Savitzky-Golay Filter (Advanced)

A rolling mean is simple, but it can flatten out interesting peaks. A more advanced method is the **Savitzky-Golay ("Savgol") filter**.

Instead of just averaging, the Savgol filter fits a small polynomial (a tiny curve) to each window of data. This is often *much* better at preserving the true shape of the sentiment arc, keeping the peaks and valleys sharp.

It has two hyperparameters:
1.  `SAVGOL_WINDOW`: How many sentences to look at (must be **odd**).
2.  `SAVGOL_POLY`: The order of the polynomial (e.g., 2 or 3). It must be *less than* the window size.

In [ ]:
# --- Savgol Hyperparameters ---
# We'll use the same window size as our rolling mean
SAVGOL_WINDOW = win_size 
SAVGOL_POLY = 3 # A cubic polynomial is a good default
# ------------------------------

print(f"Applying Savitzky-Golay filter with window {SAVGOL_WINDOW} and order {SAVGOL_POLY}...")

# Apply the Savgol filter
# Note: We must handle cases where the window is larger than the data
if len(sentiment_df) > SAVGOL_WINDOW:
    sentiment_df['vader_smooth_savgol'] = savgol_filter(sentiment_df['vader'], 
                                                      SAVGOL_WINDOW, 
                                                      SAVGOL_POLY)
    
    sentiment_df['roberta_smooth_savgol'] = savgol_filter(sentiment_df['roberta'], 
                                                      SAVGOL_WINDOW, 
                                                      SAVGOL_POLY)
else:
    print(f"Book is too short for this window size! Using rolling mean instead.")
    sentiment_df['vader_smooth_savgol'] = sentiment_df['vader_smooth_mean']
    sentiment_df['roberta_smooth_savgol'] = sentiment_df['roberta_smooth_mean']

sentiment_df.head()

---

## 📊 Step 5: Plotting and Interpreting the Arc

Now for the fun part! We'll use the `plotly` library to create an interactive plot of our smoothed data. 

**Try this:** Hover your mouse over the plot to see the exact sentence that corresponds to any point on the arc!

In [ ]:
# Create a 'percent_through_novel' column for a clear X-axis (0% to 100%)
sentiment_df['percent_through_novel'] = (sentiment_df['line_no'] / len(sentiment_df)) * 100

# "Melt" the dataframe to make it 'tidy' for Plotly.
# This stacks our two sentiment columns into one, with a new 'Model' column.
plot_df = sentiment_df.melt(
    id_vars=['percent_through_novel', 'line_no', 'line'],
    value_vars=['vader_smooth_savgol', 'roberta_smooth_savgol'],
    var_name='Model',
    value_name='Sentiment Score'
)

# Map the column names for a cleaner legend
plot_df['Model'] = plot_df['Model'].map({
    'vader_smooth_savgol': 'VADER (Lexicon)',
    'roberta_smooth_savgol': 'RoBERTa (AI Model)'
})

# --- Generate the Interactive Plot ---

print("Generating interactive plot...")

fig = px.line(plot_df, 
              x='percent_through_novel', 
              y='Sentiment Score',
              color='Model', # Create a different line for each model
              hover_data={ # Define what shows up on hover
                  'percent_through_novel': ':.1f%', 
                  'line_no': True, 
                  'line': True,
                  'Sentiment Score': ':.3f'
              },
              title=f"Sentiment Arc for '{novel_name_str}'")

# Customize the layout
fig.update_layout(
    xaxis_title="Narrative Time (Percent Through Novel)",
    yaxis_title="Sentiment Score (Smoothed)",
    legend_title="Sentiment Model",
    hovermode="x unified" # Shows hover data for all lines at once
)

fig.show()

### How to Interpret This Plot

This graph shows the emotional journey of your novel. 

* **Peaks (High Points):** Look for moments of joy, resolution, excitement, or positive events. Hover over a peak to see what sentence caused it.

* **Valleys (Low Points):** These are moments of conflict, sadness, tension, or negative events. These are often the most dramatic parts of a story.

* **Slope (Steepness):** A steep drop or sharp rise shows a sudden change in the story's emotional tone.

* **Model Differences:** Where do the VADER and RoBERTa models agree or disagree? The AI model (RoBERTa) is often better at catching sarcasm or complex sentences (like "I am not at all displeased"), which VADER might score incorrectly.

---

## Advanced Appendix: Using R-Based Lexicons (SyuzhetR & SentimentR)

This section is **optional** and preserved from the original notebook. It shows how to use the R language *inside* Python to run even more lexicon-based models from the `syuzhet` and `sentimentr` packages.

In [ ]:
# Install rpy2 to allow Python to talk to R
!pip install -q rpy2

In [ ]:
%load_ext rpy2.ipython

import rpy2.robjects as robjects
from rpy2.robjects.packages import importr
import rpy2.robjects.numpy2ri
rpy2.robjects.numpy2ri.activate()

In [ ]:
%%time
%%capture
%%R

# Install Syuzhet.R, Sentiment.R and Utility Libraries
install.packages(c('syuzhet', 'sentimentr', 'tidyverse', 'lexicon'))

library(syuzhet)
library(sentimentr)
library(tidyverse)
library(lexicon)

### A. SyuzhetR (4 Models)

In [ ]:
%%time

syuzhet = importr('syuzhet')

# Create a new DataFrame for these results
syuzhet_df = sentiment_df[['line_no', 'line']].copy(deep=True)
line_list_r = syuzhet_df['line'].to_list()

print('[1/4] Processing syuzhetr_syuzhet')
syuzhet_df['syuzhetr_syuzhet'] = syuzhet.get_sentiment(line_list_r, method='syuzhet')
print('[2/4] Processing syuzhetr_bing')
syuzhet_df['syuzhetr_bing'] = syuzhet.get_sentiment(line_list_r, method='bing')
print('[3/4] Processing syuzhetr_afinn')
syuzhet_df['syuzhetr_afinn'] = syuzhet.get_sentiment(line_list_r, method='afinn')
print('[4/4] Processing syuzhetr_nrc')
syuzhet_df['syuzhetr_nrc'] = syuzhet.get_sentiment(line_list_r, method='nrc')

syuzhet_df.head()

In [ ]:
print("Plotting SyuzhetR model results...")
# Apply smoothing
win_size_r = int(len(syuzhet_df) * 0.05)
if win_size_r % 2 == 0: win_size_r += 1
    
syuzhet_model_ls = ['syuzhetr_syuzhet', 'syuzhetr_bing', 'syuzhetr_afinn', 'syuzhetr_nrc']
for col in syuzhet_model_ls:
    syuzhet_df[col] = syuzhet_df[col].rolling(win_size_r, center=True, min_periods=1).mean()

# Plot
fig = px.line(syuzhet_df, x='line_no', y=syuzhet_model_ls, title='SyuzhetR Model Sentiment Arcs')
fig.show()

### B. SentimentR (8 Models)

In [ ]:
%%file get_sentimentr.R

library(sentimentr)
library(lexicon)

get_sentimentr_values <- function(s_v) {

  print('[1/8] Processing sentimentr_jockersrinker')
  sentimentr_jockersrinker <- sentiment(s_v, polarity_dt=lexicon::hash_sentiment_jockers_rinker,
                                        hypen="", amplifier.weight=0.8, n.before=5, n.after=2,
                                        adversative.weight=0.25, neutral.nonverb.like = FALSE, missing_value = 0)

  print('[2/8] Processing sentimentr_jockers')
  sentimentr_jockers <- sentiment(s_v, polarity_dt=lexicon::hash_sentiment_jockers,
                                        hypen="", amplifier.weight=0.8, n.before=5, n.after=2,
                                        adversative.weight=0.25, neutral.nonverb.like = FALSE, missing_value = 0)

  print('[3/8] Processing sentimentr_huliu')
  sentimentr_huliu <- sentiment(s_v, polarity_dt=lexicon::hash_sentiment_huliu,
                                        hypen="", amplifier.weight=0.8, n.before=5, n.after=2,
                                        adversative.weight=0.25, neutral.nonverb.like = FALSE, missing_value = 0)

  print('[4/8] Processing sentimentr_nrc')
  sentimentr_nrc <- sentiment(s_v, polarity_dt=lexicon::hash_sentiment_nrc,
                                        hypen="", amplifier.weight=0.8, n.before=5, n.after=2,
                                        adversative.weight=0.25, neutral.nonverb.like = FALSE, missing_value = 0)

  print('[5/8] Processing sentimentr_senticnet')
  sentimentr_senticnet <- sentiment(s_v, polarity_dt=lexicon::hash_sentiment_senticnet,
                                        hypen="", amplifier.weight=0.8, n.before=5, n.after=2,
                                        adversative.weight=0.25, neutral.nonverb.like = FALSE, missing_value = 0)

  print('[6/8] Processing sentimentr_sentiword')
  sentimentr_sentiword <- sentiment(s_v, polarity_dt=lexicon::hash_sentiment_sentiword,
                                        hypen="", amplifier.weight=0.8, n.before=5, n.after=2,
                                        adversative.weight=0.25, neutral.nonverb.like = FALSE, missing_value = 0)

  print('[7/8] Processing sentimentr_loughran_mcdonald')
  sentimentr_loughran_mcdonald <- sentiment(s_v, polarity_dt=lexicon::hash_sentiment_loughran_mcdonald,
                                        hypen="", amplifier.weight=0.8, n.before=5, n.after=2,
                                        adversative.weight=0.25, neutral.nonverb.like = FALSE, missing_value = 0)

  print('[8/8] Processing sentimentr_socal_google')
  sentimentr_socal_google <- sentiment(s_v, polarity_dt=lexicon::hash_sentiment_socal_google,
                                        hypen="", amplifier.weight=0.8, n.before=5, n.after=2,
                                        adversative.weight=0.25, neutral.nonverb.like = FALSE, missing_value = 0)

  anovel_sentimentr_df <- data.frame(
                                'sentimentr_jockersrinker' = sentimentr_jockersrinker$sentiment,
                                'sentimentr_jockers' = sentimentr_jockers$sentiment,
                                'sentimentr_huliu' = sentimentr_huliu$sentiment,
                                'sentimentr_nrc' = sentimentr_nrc$sentiment,
                                'sentimentr_senticnet' = sentimentr_senticnet$sentiment,
                                'sentimentr_sentiword' = sentimentr_sentiword$sentiment,
                                'sentimentr_loughran_mcdonald' = sentimentr_loughran_mcdonald$sentiment,
                                'sentimentr_socal_google' = sentimentr_socal_google$sentiment
                                )
  return(anovel_sentimentr_df)

}

In [ ]:
# Setup python robject with external R function
r = robjects.r

# Loading the function we have defined in R.
r['source']('get_sentimentr.R')

# Get the R function into our Python environment
get_sentimentr_function_r = robjects.globalenv['get_sentimentr_values']

In [ ]:
%%time

print("Running SentimentR (8 models)...")
line_ls = sentiment_df['line'].to_list()

# Convert Python List of Strings to a R vector of characters
sentence_v = robjects.StrVector(line_ls)
sentiment_df_r = get_sentimentr_function_r(sentence_v)

# Convert rpy2.robjects.vectors.DataFrame to pandas.core.frame.DataFrame
from rpy2.robjects import pandas2ri
with (ro.default_converter + pandas2ri.converter).context():
  sentimentr_df = ro.conversion.get_conversion().rpy2py(sentiment_df_r)

sentimentr_df['line_no'] = sentimentr_df.index
print("SentimentR analysis complete.")
sentimentr_df.head()

In [ ]:
print("Plotting SentimentR model results...")
# Apply smoothing
win_size_r = int(len(sentimentr_df) * 0.05)
if win_size_r % 2 == 0: win_size_r += 1
    
sentimentr_model_ls = list(sentimentr_df.columns.drop('line_no'))
for col in sentimentr_model_ls:
    sentimentr_df[col] = sentimentr_df[col].rolling(win_size_r, center=True, min_periods=1).mean()

# Plot
fig = px.line(sentimentr_df, x='line_no', y=sentimentr_model_ls, title='SentimentR Model Sentiment Arcs')
fig.show()